# Qwen Voice Studio — Final Narration Generator

A private-by-password text-to-speech studio powered by Qwen3-TTS from Hugging Face. It runs on a Kaggle GPU and exposes a temporary interface through Cloudflare Quick Tunnel. This notebook is configured for the LINE 13 first-person narration, with Ryan selected and a low, resonant male delivery at a brisk pace.

**Before running:** enable Internet and a GPU accelerator in Kaggle settings. The first run installs packages and downloads the model. The last cell generates a temporary login and tunnel URL; keep both private and stop the notebook session when finished.

In [ ]:
%pip -q install -U qwen-tts gradio soundfile
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
print('Qwen3-TTS and Cloudflare Tunnel are ready.')

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Notebook settings.")

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"

tts = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.float32,
    attn_implementation="sdpa",
)

speakers = tts.get_supported_speakers()
languages = tts.get_supported_languages()
print(f"Model ready on {torch.cuda.get_device_name(0)}")

In [ ]:
import secrets
import socket
import subprocess
import tempfile
import threading
import time
import uuid
from pathlib import Path

import gradio as gr
import soundfile as sf

MAX_CHARACTERS = 1800

# Ask the OS for an available local port.
with socket.socket() as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    PORT = port_probe.getsockname()[1]


def generate_speech(text, speaker, language, direction):
    text = (text or "").strip()
    if not text:
        raise gr.Error("Write a line of text before generating speech.")
    if len(text) > MAX_CHARACTERS:
        raise gr.Error(f"Keep each generation under {MAX_CHARACTERS:,} characters.")

    with torch.inference_mode():
        wavs, sample_rate = tts.generate_custom_voice(
            text=text,
            language=language,
            speaker=speaker,
            instruct=(direction.strip() or None),
        )

    output = Path(tempfile.gettempdir()) / f"qwen-voice-{uuid.uuid4().hex}.wav"
    sf.write(output, wavs[0], sample_rate)
    return str(output), str(output)


css = """
#studio {max-width: 1120px; margin: 0 auto;}
.hero {padding: 24px 4px 12px;}
.hero h1 {font-size: 2.55rem; letter-spacing: -0.05em; margin: 0 0 6px;}
.hero p {font-size: 1rem; opacity: .76;}
.generate button {min-height: 52px; font-size: 1rem; font-weight: 700;}
"""
theme = gr.themes.Soft(primary_hue="violet", secondary_hue="slate")

with gr.Blocks(title="Qwen Voice Studio") as demo:
    with gr.Column(elem_id="studio"):
        gr.HTML(
            '<div class="hero"><h1>Qwen Voice Studio</h1>'
            "<p>Write a line. Choose a voice. Hear it come alive.</p></div>"
        )

        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                script = gr.Textbox(
                    label="Your script",
                    placeholder="Paste or write something to say …",
                    lines=9,
                    max_lines=18,
                    max_length=MAX_CHARACTERS,
                    value=(
                        'I moved to Morrow Creek for a job that was supposed to be boring '
                        'The power company needed a night technician '
                        'I needed a fresh start '
                        'The town had one diner, one water tower, and windows that went dark before ten '
                        'At 1:13 a.m '
                        'on my first shift, the outage board flashed '
                        'One customer was drawing power from Line 13 '
                        'Fourteen Old Bell Road '
                        'Dispatch said, “That road isn’t in our system.” My truck GPS showed a gray line leading past the last streetlight '
                        'At the end stood a house with one upstairs window lit '
                        'The meter outside was turning backwards '
                        'I stepped out to read the service tag '
                        'Mae came over the radio: “Eli, get back in the truck.” Then my own voice came through, breathless and terrified '
                        '“Don’t shut it off '
                        'It’s already inside.” The upstairs curtain moved '
                        'A little girl pressed three fingers to the glass '
                        'Behind her stood a man in my work jacket, holding my radio '
                        'He leaned close to the window and smiled '
                        '“You came early this time.” Every light in the house went out '
                        'The outage board refreshed '
                        'A second customer had connected to Line 13 '
                        'Twelve Foundry Lane '
                        'My new address '
                        'I hadn’t told anyone where I was staying.'
                    ),
                )
                count = gr.Markdown("**0** characters")
                script.input(
                    lambda value: f"**{len(value or '')}** characters",
                    inputs=script,
                    outputs=count,
                )

                with gr.Row():
                    voice = gr.Dropdown(
                        speakers,
                        value="Ryan" if "Ryan" in speakers else speakers[0],
                        label="Voice",
                    )
                    language = gr.Dropdown(
                        languages,
                        value="English" if "English" in languages else languages[0],
                        label="Language",
                    )

                direction = gr.Textbox(
                    label="Delivery direction (optional)",
                    placeholder="For example: Warm and reassuring, with a gentle pace",
                    lines=3,
                    value=(
                        "Low, resonant male baritone; mature and close-miked. Speak briskly "
                        "with tight timing, quick natural pacing, and short pauses. Clear English, "
                        "first person. Start casual, then grow tense; drop to a controlled near-whisper "
                        "on ‘It’s already inside’ and ‘You came early this time.’ No music or sound effects."
                    ),
                )
                create = gr.Button(
                    "Generate speech",
                    variant="primary",
                    elem_classes=["generate"],
                )

            with gr.Column(scale=5):
                gr.Markdown(
                    "### Your audio\n"
                    "The first generation may take a little longer while the model warms up."
                )
                preview = gr.Audio(label="Preview", type="filepath", interactive=False)
                download = gr.File(label="Download WAV", interactive=False)
                gr.Markdown(
                    "Tip: A speaker’s native language usually gives the most natural result."
                )

        create.click(
            generate_speech,
            inputs=[script, voice, language, direction],
            outputs=[preview, download],
        )

ACCESS_PASSWORD = secrets.token_urlsafe(12)

demo.launch(
    server_name="127.0.0.1",
    server_port=PORT,
    share=False,
    auth=("voice", ACCESS_PASSWORD),
    show_error=True,
    prevent_thread_lock=True,
    theme=theme,
    css=css,
)

tunnel = subprocess.Popen(
    [
        "/kaggle/working/cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


def show_tunnel_logs():
    for line in tunnel.stdout:
        print(line.rstrip(), flush=True)


threading.Thread(target=show_tunnel_logs, daemon=True).start()

print("Studio login username: voice")
print("Studio access password (keep private):", ACCESS_PASSWORD)
print("The temporary https://…trycloudflare.com URL will appear in the tunnel logs above.")
print("Keep this cell running. Stop the Kaggle session to close the public URL.")

## Access and safety

Cloudflare Quick Tunnels are temporary demo URLs. The interface requires the username `voice` and a random password generated for this session. Keep both the URL and password private. Requests travel through Cloudflare Tunnel to the Kaggle runtime; speech generation runs in this notebook's GPU session. Stop the notebook session to end access.

Model: [Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice) · [Qwen3-TTS usage guide](https://github.com/QwenLM/Qwen3-TTS) · [Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)